# Level 2b — From cell types to tissue architecture

### CAJAL Neuromics 2026 · Project 15: Mapping the Spatial Cellular Architecture of the Brain

**~ ½ day · graded**

> *"We know what the cells are. Now: where are they, how are they arranged, and how do they talk?"*

In **2a** you put a trustworthy cell-type label on every MERFISH cell by reference-mapping from the
multiome atlas. This notebook reads *where* those cells sit and *how* neighbouring types communicate —
reproducing the core of **Wang et al. 2025, Figure 2**. The biological arc is the paper's own:

1. **cell types in space** → 2. **niches = the cortical cytoarchitecture** → 3. **neighbourhood
enrichment** (who sits next to whom) → 4. **cell–cell communication** (who signals to whom).

**Learning objectives — by the end you can:**
- Plot annotated cells in physical space and read the developing cortex off the map.
- Define **spatial niches** by clustering each cell's neighbourhood cell-type composition (CLR + k-means)
  and recognise them as the ventricular zone → intermediate zone → cortical layers → white matter.
- Quantify spatial co-localisation with **neighbourhood enrichment** (`squidpy.gr.nhood_enrichment`).
- Infer **cell–cell communication** by cell type (LIANA) and recover the paper's neuregulin /
  somatostatin signalling — and critically compare to the authors' results.

**How to work through this notebook**
- 🔬 **TASK** / 💡 **HINT** / ❓ **QUESTION** / ⚠️ **CHECKPOINT** as before. **Kernel:** `Spatial Brain (SIF)`.
- Yellow **`FutureWarning`** boxes are normal (not errors). Some cells run for **a few minutes**
  (spatial graph, niche clustering, reference mapping) — a slow cell is working, not hung.
- **Data:** reads your own `data/wang2025_merfish/processed/cohort_annotated.h5ad` from 2a and the
  shared read-only RNA reference. A ≥ 32 GB node is recommended.


---
## 0. Setup & load 2a's outputs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scanpy as sc
import squidpy as sq
import seaborn as sns

sc.settings.verbosity = 1
sc.set_figure_params(dpi=80, frameon=False, figsize=(4, 4))
%matplotlib inline

DATA = "/shared/projects/tp_2630_ubordeaux_neuromics_184418/projects/C15/data"
REF_PATH = f"{DATA}/wang2025_multiome/processed/wang2025_multiome_rna.h5ad"
GROUND_TRUTH_PATH = f"{DATA}/wang2025_merfish/processed/wang2025_merfish_cells.h5ad"

from spatialbrain import FilePaths

proc = FilePaths.dataset("wang2025_merfish").processed
cohort = sc.read_h5ad(proc / "cohort_annotated.h5ad")  # 2a §11: labels + measured genes
lr_imputed = sc.read_h5ad(proc / "cohort_lr_imputed.h5ad")  # 2a §11: imputed ligand-receptor genes
cohort.obs["type_pred"] = cohort.obs["type_pred"].astype("category")
samples = cohort.obs["Sample_ID"].cat.categories.tolist()
print(cohort)
print("\nimputed LR object for communication:", lr_imputed.shape)

---
## 1. Cell types in space (Fig. 2a)

Plot **all six sections** in physical space, coloured by the cell type you assigned in 2a. Even by eye
you should see **layering** — progenitors on the ventricular side, neurons stacked outward — and how it
differs between PFC and V1 and across age.

🔬 **TASK 1.1 — Plot the whole cohort in space.** Draw all six sections coloured by `type_pred`.

💡 **HINT:** `sq.pl.spatial_scatter` with `library_key="Sample_ID"`, `library_id=samples`, `shape=None`
and a small `size` renders hundreds of thousands of cells fast.

In [ ]:
# YOUR CODE HERE — see the hint above
...

❓ **QUESTION.** Pick a section: can you see a radial gradient of cell types (progenitors → mature
neurons)? That gradient is the developing cortex's radial organisation — which we formalise as *niches*
next.

---
## 2. The spatial graph — build it, then *see* the tissue

Everything spatial (niches, neighbourhood enrichment) rests on a **spatial neighbour graph**: which
cells are physically close. We build it **per sample** (`library_key`) so edges never cross two separate
sections. Then we do two things: (1) render **one entire section** at high resolution — the cell-type
map *is* the tissue architecture, and you see far more structure than in any crop; (2) zoom into a small
crop and draw the graph edges, to confirm they connect physically adjacent cells.

🔬 **TASK 2.1 — Build the graph; show a full section and a zoomed graph crop.** Build a per-sample
50-nearest-neighbour graph on the whole cohort (this is what the niche step needs). Draw one full
section coloured by `type_pred`; then, on a small coordinate window of that section, draw a Delaunay
graph so the edges are legible.

💡 **HINT:** `sq.gr.spatial_neighbors(coord_type="generic", n_neighs=50, library_key="Sample_ID")`.
For the crop, subset one `Sample_ID` to a coordinate window a few hundred µm wide, rebuild with
`delaunay=True`, and pass `connectivity_key="spatial_connectivities"` to `sq.pl.spatial_scatter`.

In [ ]:
# analysis graph: per-sample 50 nearest neighbours (used by the niche step below)
# one FULL section — the cell-type map is the tissue architecture


# YOUR CODE HERE — see the hint above
...

In [ ]:
# zoom into a small crop and draw the graph edges to confirm they connect nearby cells


# YOUR CODE HERE — see the hint above
...

❓ **QUESTION.** In the full section, can you trace the ventricular zone → intermediate zone →
cortical plate by cell type alone? In the crop, is each cell linked to *physically nearby* cells? If the
edges spanned the whole tissue, something would be wrong with the coordinates — which is why we look.

---
## 3. Spatial niches = the cortical cytoarchitecture (Fig. 2a/b)

A **niche** is a recurring local cellular neighbourhood. The authors defined 10 niches by **k-means on
the cell-type composition of each cell's 50 nearest spatial neighbours** — and they turned out to be the
classic cortical domains (VZ/SVZ, intermediate zone, cortical layers 1–6, white matter).

We do the same, with one statistical improvement. Each neighbourhood is a vector of **cell-type
proportions** — *compositional* data on the simplex. Plain Euclidean k-means on proportions is not
ideal; the principled fix is a **centred-log-ratio (CLR) transform** first, so Euclidean distance
becomes **Aitchison distance**. The composition is just `graph @ one-hot(type)` — fast on all 404k
cells — so we cluster **all samples together** into shared niches.

🔬 **TASK 3.1 — Call the niches.** Build the neighbour cell-type composition, CLR-transform it,
and k-means into 10 niches.

In [ ]:
# 1. neighbour cell-type composition: for each cell, the fraction of its 50 neighbours of each type
# 2. CLR transform (pseudocount for the many zero proportions) -> Aitchison geometry
# 3. k-means into 10 niches (fixed k, like the paper)


# YOUR CODE HERE — see the hint above
...

⚠️ **CHECKPOINT.** Exactly **10 niches**. Now *see* them: niches in space beside the cell types they're
built from, and each niche's cell-type composition.

In [ ]:
# niches beside cell types in space (Fig. 2a) — two sections, type vs niche side by side
sq.pl.spatial_scatter(
    cohort,
    library_key="Sample_ID",
    library_id=samples[:2],
    color=["type_pred", "niche"],
    shape=None,
    size=2,
    ncols=2,
    figsize=(13, 11),
)
# all six sections, niches only
sq.pl.spatial_scatter(
    cohort,
    library_key="Sample_ID",
    library_id=samples,
    color="niche",
    shape=None,
    size=2,
    ncols=3,
    figsize=(16, 9),
)

In [ ]:
# composition: proportion of each cell type within each niche (Fig. 2b)
comp = pd.crosstab(cohort.obs["niche"], cohort.obs["type_pred"], normalize="index")
plt.figure(figsize=(14, 5))
sns.heatmap(comp, cmap="viridis")
plt.title("cell-type composition per niche")
plt.show()

🔬 **TASK 3.2 — Name the niches biologically.** From the composition heatmap and the side-by-side
spatial plots, give each niche a cortical-domain label. Which niche is progenitor-rich (**VZ/SVZ**)?
Which are single-layer excitatory (the **cortical layers**)? Which is oligodendrocyte-rich (**white
matter**)?

💡 **HINT:** progenitors = RG-*/IPC-*; deep layers = EN-L5/L6; upper layers = EN-L2_3/L4; white
matter = Oligodendrocyte-rich. `IN-dLGE-Immature` appears both in its own dorsal-LGE niche and dispersed
in white matter (future interstitial interneurons) — a distinct 10th domain, don't fold it into "white
matter".

🚀 **Going further (optional).** Are these niches robust or a parameter artefact? Re-call for a few
`n_neighs`/`k` and measure partition agreement (`sklearn.metrics.adjusted_rand_score`).

---
## 4. Neighbourhood enrichment — who sits next to whom (Fig. 2c)

Niches show *domains*; neighbourhood enrichment tests, pairwise, which cell **types** are spatial
neighbours more (or less) than chance. Fig. 2c is the **PFC section at infancy**, run **per sample**, so
we pick that section and a *tight* graph (`n_neighs=6` — direct adjacency, not the broad 50 used for
niches). To read it like the paper, we **order the matrix by cell class** (progenitor / EN / IN / glia /
vascular) and hierarchically cluster within each class — the block structure then jumps out: excitatory
types cluster by layer, and you can look for **EN↔IN** cross-blocks.

🔬 **TASK 4.1 — Enrichment on the PFC-infancy section, ordered by class.** Subset to the oldest PFC
sample, build a tight graph, run `sq.gr.nhood_enrichment`, then reorder the z-score matrix by class and
draw it with class colour bars.

💡 **HINT:** the z-scores are in `pfc_ad.uns["type_pred_nhood_enrichment"]["zscore"]`. Get a
type→class map from `cohort.obs` (majority `class_pred` per `type_pred`), group types by class, and
hierarchically cluster within each block (`scipy.cluster.hierarchy`). `sns.clustermap(..., row_cluster=
False, col_cluster=False, row_colors=..., col_colors=...)` draws it with the class bars.

In [ ]:
# YOUR CODE HERE — see the hint above
...

In [ ]:
# type -> class (majority class_pred per type_pred, carried from 2a), then order by class


# YOUR CODE HERE — see the hint above
...

❓ **QUESTION.** Along the diagonal, do excitatory types form their own (layer-ordered) block and
inhibitory types theirs? Can you spot a positive **EN↔IN** off-diagonal patch (e.g. `EN-L4-IT ↔
IN-MGE-SST`, `EN-IT-Immature ↔ IN-CGE-VIP`)? Spatial adjacency is a *prerequisite* for the short-range
signalling we look at next.

---
## 5. (Light) the reference atlas (Fig. 1b)

A quick look at the multiome reference these labels came from — the molecular taxonomy on its own UMAP.

In [ ]:
ref = sc.read_h5ad(REF_PATH)
if "X_wnn.umap" in ref.obsm:
    ref.obsm["X_umap"] = ref.obsm["X_wnn.umap"]
    sc.pl.umap(ref, color="type", legend_loc="on data", legend_fontsize=5, size=2)

---
## 6. Cell–cell communication (Fig. 2e/f)

The 300-gene MERFISH panel is a *cell-typing* panel — it contains almost no complete **ligand–receptor
pairs** (the ligand `SST` is on-panel, its receptors are not; none of the `NRG1/2/3`–`ERBB3/4` pairs
are). So you **cannot score LR interactions from the measured spatial data alone**. In **2a** we already
solved this: the same reference mapping that gave us labels also **imputed** the ligand–receptor
transcriptome, and we validated that imputation on held-out genes. We just load `lr_imputed` and run
**LIANA** by cell type — no re-mapping here.

In [ ]:
# confirm the panel gap, and that the imputed object fills it
print("on the panel:  SST =", "SST" in cohort.var_names, "| SSTR2 =", "SSTR2" in cohort.var_names)
print(
    "imputed genes:",
    lr_imputed.n_vars,
    "| e.g.",
    [g for g in ["SSTR2", "ERBB4", "EGFR", "GRM7"] if g in lr_imputed.var_names],
)

In [ ]:
import liana as li

# log-normalise the imputed counts for LIANA (it expects log1p-CP10k in .X)
comm = lr_imputed.copy()
comm.layers["counts"] = comm.X.copy()
sc.pp.normalize_total(comm, target_sum=1e4)
sc.pp.log1p(comm)
comm.obs["type_pred"] = comm.obs["type_pred"].astype("category")
print(comm)

🔬 **TASK 6.1 — Score interactions with LIANA.** Run `li.mt.natmi` grouped by `type_pred` on
`comm`, then look at the strongest interactions overall.

💡 **HINT:** `li.mt.natmi(comm, groupby="type_pred", expr_prop=0.1, use_raw=False)` writes
`comm.uns["liana_res"]`. Sort by `expr_prod` (interaction magnitude).

In [ ]:
# YOUR CODE HERE — see the hint above
...

Now *visualise* the communication. LIANA's `dotplot` shows, per **sender**, the ligand→receptor
interactions (rows) across **targets** (columns) — colour = magnitude (`expr_prod`), size = specificity
(`spec_weight`). We focus on the paper's two senders.

In [ ]:
li.pl.dotplot(
    adata=comm,
    colour="expr_prod",
    size="spec_weight",
    source_labels=["IN-MGE-SST", "EN-IT-Immature"],
    top_n=25,
    orderby="expr_prod",
    orderby_ascending=False,
    figure_size=(10, 8),
    return_fig=True,
)

🔬 **TASK 6.2 — Recover the paper's signalling.** From `res`, pull interactions where the sender
is `IN-MGE-SST` with ligand `SST`/`CORT` (somatostatin), and where the sender is `EN-IT-Immature` with
ligand `NRG1/2/3` (neuregulin). Sort by `expr_prod`. Are the top targets excitatory neurons (for SST)
and interneurons/glia (for NRG)?

In [ ]:
# YOUR CODE HERE — see the hint above
...

⚠️ **CHECKPOINT.** **Somatostatin** from `IN-MGE-SST` should target excitatory neurons
(EN-L2_3/L4/L5/L6-IT, EN-IT-Immature) — Fig. 2f. **Neuregulin** from `EN-IT-Immature` should target
interneurons and glia — Fig. 2e. The *receptors* LIANA reports (`GRM7`, `EGFR`, `NETO2`) differ from the
paper's CellChatDB pairs (`SSTR`, `ERBB`) — a **resource difference**, not a biological one.

Finally, put one signalling axis **in space**. On the PFC-infancy section, show the SST-sending
interneurons and their excitatory targets as cell types, beside the imputed `SST` ligand and `GRM7`
receptor — signalling needs sender and receiver to be spatial neighbours, which the niche/enrichment
steps showed they are.

In [ ]:
from matplotlib.colors import ListedColormap

sig = cohort[cohort.obs["Sample_ID"] == pfc_s].copy()
# one categorical column: SST-sending interneurons, their EN-L4 targets, everything else
role = np.full(sig.n_obs, "other", dtype=object)
role[sig.obs["type_pred"] == "IN-MGE-SST"] = "SST sender"
role[sig.obs["type_pred"].astype(str).str.startswith("EN-L4")] = "EN-L4 target"
sig.obs["signalling"] = pd.Categorical(role, categories=["EN-L4 target", "SST sender", "other"])

# imputed ligand + receptor onto the same cells (log-normalised comm)
for g in ["SST", "GRM7"]:
    if g in comm.var_names:
        x = comm[sig.obs_names, g].X
        sig.obs[f"imp_{g}"] = np.asarray(x.todense()).ravel() if hasattr(x, "todense") else np.asarray(x).ravel()

sq.pl.spatial_scatter(
    sig,
    color="signalling",
    palette=ListedColormap(["#1f77b4", "#d62728", "0.85"]),
    shape=None,
    size=5,
    figsize=(7, 6),
    title=f"SST signalling axis — {pfc_s}",
)
sq.pl.spatial_scatter(
    sig,
    color=[c for c in ["imp_SST", "imp_GRM7"] if c in sig.obs],
    shape=None,
    size=5,
    cmap="magma",
    ncols=2,
    figsize=(13, 5),
)

---
## 7. The biological story — did we rebuild the cortex? (compare to the authors)

Load the authors' original niche labels (now allowed) and compare — but the question isn't a number,
it's biology: **did our niches recover the same cortical architecture?** The authors built their niches
*independently* (k-means on neighbourhoods of their own labels), so agreement is genuine
cross-validation. We quantify it (ARI/NMI + which of our niches maps to which named domain) **and** put
the two niche maps **side by side in space**.

In [ ]:
truth = sc.read_h5ad(GROUND_TRUTH_PATH, backed="r")
cohort.obs["niche_true"] = pd.Categorical(truth.obs.loc[cohort.obs_names, "niche_name"].values)
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(cohort.obs["niche_true"].astype(str), cohort.obs["niche"].astype(str))
nmi = normalized_mutual_info_score(cohort.obs["niche_true"].astype(str), cohort.obs["niche"].astype(str))
print(f"niche agreement vs authors: ARI={ari:.3f}  NMI={nmi:.3f}")

# which of our niches maps to which named cortical domain?
xt = pd.crosstab(cohort.obs["niche"], cohort.obs["niche_true"], normalize="index")
plt.figure(figsize=(10, 5))
sns.heatmap(xt, cmap="magma")
plt.title("our niches (rows) vs authors' named domains (cols)")
plt.show()

In [ ]:
# our niches vs the authors' named domains, side by side in space on the PFC-infancy section
sq.pl.spatial_scatter(
    cohort,
    library_key="Sample_ID",
    library_id=[pfc_s],
    color=["niche", "niche_true"],
    shape=None,
    size=3,
    ncols=2,
    figsize=(14, 7),
)

🔬 **TASK 7.1 — Write the story (3–5 sentences).** Using the crosstab and the side-by-side maps,
state which cortical domains you recovered and how cleanly they map to the authors'. Connect it to
development: progenitors in the VZ/SVZ, excitatory neurons laminated inside-out, interneurons and
oligodendrocytes in the white matter — and the reciprocal EN↔IN (neuregulin / somatostatin) signalling
that helps build it. Where did your reconstruction diverge, and is that biology or a method/database
artefact?

🚀 **Going further (optional).** We imputed the transcriptome to recover off-panel receptors. What
survives using only the **measured** panel? Re-run LIANA on `cohort` with only LR pairs whose *both*
partners are on-panel, and compare top senders/receivers to the imputed result.

---
## Where this is heading

You've reproduced the heart of Wang et al.'s Figure 2 — cell types in space, the niche cytoarchitecture,
neighbourhood enrichment, and cell–cell communication — and critically compared it to the authors'.
**Level 3** takes the brakes off: bring in the multiome **ATAC** side and ask a question the paper left
open (the fate of white-matter interneurons, or the Tri-IPC gliogenesis programme).

In [ ]:
import session_info2

session_info2.session_info(os=True, cpu=True)